# 計算量解析

**Phase 1: 基盤**

導入: 初学者が用語と計算量を説明できることを目標にします。


## 用語辞典（詳細）
- 時間計算量: 入力サイズnが増えたとき、実行時間がどのくらい増えるかの目安。例: ループ1回ならO(n)。
- 空間計算量: アルゴリズムが追加で使うメモリ量の増え方。配列を新規作成すると増えやすい。
- Big-O記法: 厳密な秒数ではなく、増え方の傾向を比較する表記。
- 最悪計算量: 一番不利な入力を想定した上限。SLAを守るときに重要。
- 平均計算量: 実運用に近い平均ケース。アクセス分布が偏ると変わる。
- 償却計算量: 1回は高コストでも、複数回なら平均で低コストになる考え方。


## 概念説明
これは何か: アルゴリズムの速度とメモリ効率を、入力サイズ基準で比較する方法。
使いどころ: どの実装を採用するか迷うとき、性能見積もりの根拠を作る。
何が良いか: 実行環境が違っても、増加傾向で公平に比較できる。
注意点: Big-Oだけで決めず、定数倍・実データ分布・実装コストも見る。


## 計算量・式
O(1): 配列末尾append(平均), dict参照(平均)
O(log n): 二分探索
O(n): 線形探索
O(n log n): 比較ソート
O(n^2): 二重ループ
O(2^n): 部分集合全探索


## 代表的な計算量ごとのコード例

下のセルでは、よく挙げられる **O(1), O(log n), O(n), O(n log n), O(n²), O(2ⁿ)** を、短いコードで対応づけます。
（`O(2ⁿ)` は n を大きくすると実行が終わらないので、ループ例では **小さい n だけ** 試してください。）


In [1]:
# --- O(1) : 入力 n に対して「回数がほぼ増えない」操作の例 ---
# 辞書のキー参照は平均で O(1)。リスト末尾への append は償却 O(1)。
d = {"user_1": "alice", "user_2": "bob"}
name = d["user_1"]  # 平均 O(1)

stack = []
stack.append("job")  # 償却 O(1)
print("O(1) 例:", name, stack)

# --- O(log n) : 毎回探索範囲が半分になる例（ソート済み配列前提） ---
import bisect
sorted_ids = list(range(0, 1000, 10))  # n 要素のソート済み列
pos = bisect.bisect_left(sorted_ids, 450)  # O(log n)
print("O(log n) bisect:", pos)

# --- O(n) : 要素を一通り見る例 ---
nums = [3, 1, 4, 1, 5]
total = 0
for x in nums:  # O(n)
    total += x
print("O(n) sum:", total)

# --- O(n log n) : 効率の良い比較ソートが多いオーダー ---
print("O(n log n) sorted:", sorted(nums))

# --- O(n²) : 二重ループで全ペアを見る素朴な例 ---
n_small = 50
pairs = 0
for i in range(n_small):
    for j in range(n_small):  # O(n²)
        pairs += 1
print("O(n²) pairs count (n=50):", pairs)

# --- O(2ⁿ) : 部分集合やビット全列挙（n が増えると指数爆発） ---
n_exp = 18  # これ以上は時間がかかりすぎるので注意
subset_count = 0
for mask in range(1 << n_exp):  # 2^n_exp 回
    subset_count += 1
print("O(2^n) iterations (n=18):", subset_count, "= 2**", n_exp)


O(1) 例: alice ['job']
O(log n) bisect: 45
O(n) sum: 14
O(n log n) sorted: [1, 1, 3, 4, 5]
O(n²) pairs count (n=50): 2500
O(2^n) iterations (n=18): 262144 = 2** 18


## 計算量が実務・コードに与える影響（ざっくり実測）

同じ「全件を何かする」でも、**O(n) と O(n²)** ではデータ量が増えたときの待ち時間が別物になります。
下のセルでは `time.perf_counter()` で、**線形ループ**と**二重ループ**の差を n を変えて比較します。
（本番では I/O やDB・ネットワークが支配的なことも多いが、**CPU内の処理だけ**のイメージを掴む用途に使います。）


In [2]:
import time


def work_on_linear(n: int) -> None:
    """O(n): 要素を1回ずつ足し合わせるだけ"""
    s = 0
    for i in range(n):
        s += i


def work_on_quadratic(n: int) -> None:
    """O(n²): 素朴な全ペア比較イメージ（実務では避けたいパターンになりやすい）"""
    c = 0
    for i in range(n):
        for j in range(n):
            c += 1


def bench(fn, n: int) -> float:
    t0 = time.perf_counter()
    fn(n)
    return time.perf_counter() - t0


sizes = [500, 1000, 2000]
print("n を増やしたときの経過時間（秒・目安）")
print("- O(n) はだいたい n に比例して伸びる")
print("- O(n²) は n が2倍でだいたい4倍近く伸びる（定数倍は無視できないが傾向として）")
print()
for n in sizes:
    t_lin = bench(work_on_linear, n)
    t_quad = bench(work_on_quadratic, n)
    print(f"n={n:4d}  O(n)={t_lin:.6f}s   O(n²)={t_quad:.6f}s   (n²/nの比のイメージ: {t_quad / max(t_lin, 1e-9):.1f}x)")

print()
print("実務での含意:")
print("- ログ件数・ユーザー数が増えると、O(n²) はすぐボトルネックになりやすい")
print("- ハッシュ・ソート・索引を使って O(n) や O(n log n) に落とせないか検討する")
print("- 最悪 O(n²) でも n が小さければ問題にならない → まず制約とデータ規模を確認する")


n を増やしたときの経過時間（秒・目安）
- O(n) はだいたい n に比例して伸びる
- O(n²) は n が2倍でだいたい4倍近く伸びる（定数倍は無視できないが傾向として）

n= 500  O(n)=0.000044s   O(n²)=0.013591s   (n²/nの比のイメージ: 309.6x)
n=1000  O(n)=0.000030s   O(n²)=0.041727s   (n²/nの比のイメージ: 1392.3x)
n=2000  O(n)=0.000107s   O(n²)=0.164872s   (n²/nの比のイメージ: 1539.1x)

実務での含意:
- ログ件数・ユーザー数が増えると、O(n²) はすぐボトルネックになりやすい
- ハッシュ・ソート・索引を使って O(n) や O(n log n) に落とせないか検討する
- 最悪 O(n²) でも n が小さければ問題にならない → まず制約とデータ規模を確認する


時間と空間、ボトルネックを考えてトレードオフを選ぶ。